# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use free open source models on Ollama. I also use paid open-source models via Groq and OpenRouter. Only pick the models you want to!
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess


In [2]:
load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

print("Checking API Keys...")
print(f"Gemini: {'✅ Loaded' if google_api_key else '❌ Missing'}")
print(f"Groq: {'✅ Loaded' if groq_api_key else '❌ Missing'}")
print(f"OpenRouter: {'✅ Loaded' if openrouter_api_key else '❌ Missing'}")

Checking API Keys...
Gemini: ✅ Loaded
Groq: ✅ Loaded
OpenRouter: ✅ Loaded


In [3]:
# 1. Setup Gemini Client (Google)
gemini = OpenAI(
    api_key=google_api_key, 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# 2. Setup Groq Client
groq = OpenAI(
    api_key=groq_api_key, 
    base_url="https://api.groq.com/openai/v1"
)

# 3. Setup OpenRouter Client
openrouter = OpenAI(
    api_key=openrouter_api_key, 
    base_url="https://openrouter.ai/api/v1"
)

In [4]:
# The absolute best free models on each platform right now:
GEMINI_MODEL = "gemini-2.5-flash"
GROQ_MODEL = "llama-3.3-70b-versatile"
OPENROUTER_MODEL = "deepseek/deepseek-r1:free" # OpenRouter's free DeepSeek model!

# Create a list for our Gradio dropdown
models = [GEMINI_MODEL, GROQ_MODEL, OPENROUTER_MODEL]

# Map each model name to the correct client
clients = {
    GEMINI_MODEL: gemini,
    GROQ_MODEL: groq,
    OPENROUTER_MODEL: openrouter
}

print(f"Loaded {len(models)} free models ready for generation!")

Loaded 3 free models ready for generation!


In [5]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-mingw32'},
 'package_managers': ['winget', 'choco'],
 'cpu': {'brand': 'Intel(R) Core(TM) i5-8250U CPU @ 1.60GHz',
  'cores_logical': 8,
  'cores_physical': 0,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (Rev2, Built by MSYS2 project) 14.2.0',
   'g++': 'g++.EXE (Rev2, Built by MSYS2 project) 14.2.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

## Overwrite this with the commands from yesterday

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [18]:
# Updated for Windows using g++
compile_command = ["g++", "main.cpp", "-o", "main.exe", "-O3"]
run_command = ["./main.exe"]

## And now, on with the main task

In [7]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [8]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [9]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [10]:
def port(model_name, python_code):
    # Grab the correct client (Gemini, Groq, or OpenRouter) based on the model chosen
    client = clients[model_name]
    
    print(f"Sending code to {model_name}...")
    
    # Make the API call
    response = client.chat.completions.create(
        model=model_name, 
        messages=messages_for(python_code)
    )
    
    # Clean up the output
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp\n', '').replace('```cpp', '').replace('```', '')
    
    # Save it to main.cpp
    write_output(reply)
    
    return reply

In [11]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [12]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [13]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [22]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Sending code to gemini-2.5-flash...


In [24]:
compile_and_run()

Result: 3.141592656089
Execution Time: 0.883944 seconds

Result: 3.141592656089
Execution Time: 1.143737 seconds

Result: 3.141592656089
Execution Time: 1.073371 seconds



deepseek/deepseek-r1:free: Fail

gemini-2.5-flash: 0.858208 

llama-3.3-70b-versatile: 0.899372



